In [32]:
import pandas as pd 

train_df = pd.read_csv("datasets/titanic/train.csv")
train_df["train"]=True
test_df = pd.read_csv("datasets/titanic/test.csv")
test_df["train"]=False

In [33]:
print(train_df.shape)
train_df.head(10)

(891, 13)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,train
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,True
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,True
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,True
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,True
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,True
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,True
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S,True
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S,True
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,True


In [34]:
print(test_df.shape)
test_df.head(10)

(418, 12)


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,train
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q,False
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S,False
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q,False
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S,False
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S,False
5,897,3,"Svensson, Mr. Johan Cervin",male,14.0,0,0,7538,9.2250,NaN,S,False
6,898,3,"Connolly, Miss. Kate",female,30.0,0,0,330972,7.6292,NaN,Q,False
7,899,2,"Caldwell, Mr. Albert Francis",male,26.0,1,1,248738,29.0000,NaN,S,False
8,900,3,"Abrahim, Mrs. Joseph (Sophie Halaut Easu)",female,18.0,0,0,2657,7.2292,NaN,C,False
9,901,3,"Davies, Mr. John Samuel",male,21.0,2,0,A/4 48871,24.1500,NaN,S,False


In [35]:
merged_df = pd.concat([train_df,test_df],axis=0)
print(merged_df.shape)

(1309, 13)


In [36]:
merged_df.isna().sum()

PassengerId       0
Survived        418
Pclass            0
Name              0
Sex               0
Age             263
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
train             0
dtype: int64

In [37]:
merged_copy = merged_df.copy()
merged_copy = merged_df.drop(["Survived","PassengerId","Name","train"],axis=1)

In [39]:
merged_copy.head()

,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,female,35.0,1,0,113803,53.1000,C123,S
4,3,male,35.0,0,0,373450,8.0500,NaN,S


In [55]:
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

merged_copy["Embarked"] = merged_copy["Embarked"].fillna(merged_copy["Embarked"].mode()[0])
imputer_cols = ["Age","Fare"]
encoder_cols = ["Sex","Embarked","Ticket"]

num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="mean")),
    ("scaler",StandardScaler()),])
cat_pipeline = Pipeline([
    ("encoder", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

column_transformer = ColumnTransformer([
    ("numbers",num_pipeline,imputer_cols),
    ("texts",cat_pipeline,encoder_cols)
], remainder='passthrough')

In [56]:
features = merged_copy.columns.tolist()
features.remove("Cabin")
features

['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']

In [57]:
cabin_train_df = merged_copy[merged_copy["Cabin"].notna()]
cabin_test_df = merged_copy[merged_copy["Cabin"].isna()]

In [58]:
cabin_train_df.head()

,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,female,35.0,1,0,113803,53.1000,C123,S
6,1,male,54.0,0,0,17463,51.8625,E46,S
10,3,female,4.0,1,1,PP 9549,16.7000,G6,S
11,1,female,58.0,0,0,113783,26.5500,C103,S


In [59]:
X_train = cabin_train_df[features]
y_train = cabin_train_df["Cabin"].values

X_test = cabin_test_df[features]
y_test = cabin_test_df["Cabin"].values

In [60]:
X_train_scaled = column_transformer.fit_transform(X_train)
X_test_scaled = column_transformer.transform(X_test)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()
rfc.fit(X_train_scaled, y_train)

In [64]:
y_hat = rfc.predict(X_test_scaled)

In [65]:
y_hat

array(['F E57', 'B77', 'F G63', ..., 'F G63', 'F G63', 'F E69'],
      shape=(1014,), dtype=object)

In [67]:
cabin_test_df.loc[:, "Cabin"] = y_hat
merged_copy.loc[merged_copy["Cabin"].isna(), "Cabin"] = y_hat

In [73]:
merged_df["Cabin"] = merged_copy["Cabin"]
merged_df.isna().sum()

PassengerId      0
Survived       418
Pclass           0
Name             0
Sex              0
Age            263
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin            0
Embarked         2
train            0
dtype: int64

In [79]:
survive_df = merged_df.copy()

survive_df = survive_df.drop(["PassengerId","Name"],axis=1)

In [80]:
train = survive_df[merged_df["train"] == True]
test = survive_df[merged_df["train"] == False]

In [81]:
train.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,train
0,0.0,3,male,22.0,1,0,A/5 21171,7.2500,F E57,S,True
1,1.0,1,female,38.0,1,0,PC 17599,71.2833,C85,C,True
2,1.0,3,female,26.0,0,0,STON/O2. 3101282,7.9250,B77,S,True
3,1.0,1,female,35.0,1,0,113803,53.1000,C123,S,True
4,0.0,3,male,35.0,0,0,373450,8.0500,F G63,S,True


In [82]:
test.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,train
0,NaN,3,male,34.5,0,0,330911,7.8292,B77,Q,False
1,NaN,3,female,47.0,1,0,363272,7.0000,B77,S,False
2,NaN,2,male,62.0,0,0,240276,9.6875,E101,Q,False
3,NaN,3,male,27.0,0,0,315154,8.6625,F G63,S,False
4,NaN,3,female,22.0,1,1,3101298,12.2875,G6,S,False


In [83]:
features = train.columns.tolist()
features.remove("Survived")
features

['Pclass',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked',
 'train']

In [91]:
X_train = train[features]
y_train = train["Survived"]

X_test = test[features]
y_test = test["Survived"]

In [92]:
le = LabelEncoder()
le.fit(pd.concat([X_train["Cabin"], X_test["Cabin"]]))
X_train["Cabin"] = le.transform(X_train["Cabin"])
X_test["Cabin"] = le.transform(X_test["Cabin"])

X_train_scaled = column_transformer.fit_transform(X_train)
X_test_scaled = column_transformer.transform(X_test)

In [93]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()
rfc.fit(X_train_scaled, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [95]:
y_hat_survive = rfc.predict(X_test_scaled)

In [97]:
X_test["Survived"] = y_hat_survive

In [98]:
X_test.head()

,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,train,Survived
0,3,male,34.5,0,0,330911,7.8292,53,Q,False,0.0
1,3,female,47.0,1,0,363272,7.0000,53,S,False,0.0
2,2,male,62.0,0,0,240276,9.6875,148,Q,False,0.0
3,3,male,27.0,0,0,315154,8.6625,178,S,False,0.0
4,3,female,22.0,1,1,3101298,12.2875,184,S,False,1.0


In [99]:
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": y_hat_survive.astype(int)
})
submission.to_csv("submission.csv", index=False)